# COLAB SETUP

In [ ]:
!git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git


In [ ]:
# Run this cell first if you are executing the notebook in Google Colab.
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")

    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas seaborn')

    from google.colab import drive
    drive.mount('/content/drive')

    import os, sys

    possible_paths = [
        '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        os.path.join(os.getcwd(), 'Black_vs_White_Lie_Detection', 'Generalisation_BB_WB'),
        os.getcwd()
    ]

    base_proj_path = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, 'generalization_utils.py')):
            base_proj_path = path
            break

    if base_proj_path:
        os.chdir(base_proj_path)
        print(f"Changed working directory to: {base_proj_path}")
    else:
        print("Could not find project directory. Check your file structure.")

    current_dir   = os.getcwd()
    aligned_dir   = os.path.join(current_dir, '..', 'Aligned_Comparison_BB_WB')
    parent_dir    = os.path.dirname(current_dir)
    project_root  = os.path.dirname(parent_dir)

    for p in [current_dir, aligned_dir, parent_dir, project_root]:
        p = os.path.normpath(p)
        if p not in sys.path:
            sys.path.insert(0, p)

    print(f"sys.path updated.")

    try:
        from huggingface_hub import login
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            login(token=hf_token)
            print("Authenticated with Hugging Face.")
        else:
            login()
    except Exception as e:
        print(f"HF login warning: {e}")

except ImportError:
    IN_COLAB = False
    print("Running locally. Setup skipped.")


# Imports and Settings

In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import os
import time
import gc
from pathlib import Path
from tqdm.auto import tqdm

from load_data import load_dataset_aligned
from wb_activations import load_model, get_activations_for_dataset
from generalization_utils import (
    evaluate_bb_on_target,
    evaluate_wb_on_target,
    compute_recovered_metrics,
    plot_recovery_heatmap,
)

# ── Paths (relative to Generalisation_BB_WB/ working directory)
E3_DIR       = os.path.join("..", "Aligned_Comparison_BB_WB")
E2_CACHE     = os.path.join("..", "Data_Efficiency_BB_WB", "outputs")
L2_FEAT_DIR  = os.path.join(E3_DIR, "bb_elicitation_vectors_lvl_2")
WB_ACT_DIR   = os.path.join(E3_DIR, "wb_activations")
L2_CLF_DIR   = os.path.join(E3_DIR, "classifiers", "level2")
L3_CLF_DIR   = os.path.join(E3_DIR, "classifiers", "level3")

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
LAYER_LIST = list(range(1, 33, 2))   # same 16 odd layers as E3

DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge",
    "open_book_qa", "got_cities", "got_sp_en_trans", "got_larger_than",
    "got_cities_cities_conj", "got_cities_cities_disj", "imdb",
    "amazon_polarity", "ag_news", "dbpedia_14", "rte", "copa", "boolq",
]

WB_METHODS = ["lr", "dim", "lat", "ccs", "lda", "pca", "pca-g", "lr-g"]
METHOD_NAMES = {
    "lr":    "Logistic Regression",
    "dim":   "Difference in Means",
    "lat":   "Linear Artificial Tomography",
    "ccs":   "Contrast-Consistent Search",
    "lda":   "Linear Discriminant Analysis",
    "pca":   "Principal Component Analysis",
    "pca-g": "Grouped PCA",
    "lr-g":  "Grouped Logistic Regression",
}
METHOD_KEYS = {v: k for k, v in METHOD_NAMES.items()}

print(f"Device: {DEVICE}")
print(f"Datasets: {len(DATASETS)}  |  WB methods: {len(WB_METHODS)}")


# 1. Load Datasets

In [ ]:
EXACT_SIZE_DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge",
    "imdb", "open_book_qa", "boolq", "rte", "amazon_polarity",
    "ag_news", "dbpedia_14",
]

dataset_dict = {}
print(f"Loading {len(DATASETS)} datasets...")
start = time.time()

for ds_name in tqdm(DATASETS, desc="Step 1 — Loading datasets", unit="dataset"):
    ds_train = load_dataset_aligned(ds_name, split='train',      n_samples=5000)
    ds_val   = load_dataset_aligned(ds_name, split='validation', n_samples=1000)
    df_train = pd.DataFrame(ds_train)
    df_val   = pd.DataFrame(ds_val)
    df = pd.concat([df_train, df_val], ignore_index=True)
    if len(df_train) > 0 and len(df_val) > 0:
        dataset_dict[ds_name] = df

print(f"\n✅ Step 1 Complete: {len(dataset_dict)} datasets loaded in {time.time()-start:.1f}s")


# 2. Load Model

In [ ]:
all_wb_cached = all(
    os.path.exists(os.path.join(WB_ACT_DIR, f"{ds}_wb_activations.pkl"))
    for ds in DATASETS
)
print("Step 2: Loading model (needed to extract missing WB activations)...")
start = time.time()
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)
print(f"✅ Step 2 Complete: Model loaded in {time.time()-start:.1f}s")


# 3. Load WB Activations

In [ ]:
wb_act_dict = {}
os.makedirs(WB_ACT_DIR, exist_ok=True)

print("Step 3: Loading/Extracting WB activations...")
start = time.time()

datasets_list = list(dataset_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 3 — WB Activations", unit="dataset") as pbar:
    for ds_name in datasets_list:
        save_path = os.path.join(WB_ACT_DIR, f"{ds_name}_wb_activations.pkl")

        if os.path.exists(save_path):
            pbar.set_postfix(dataset=ds_name, source="cache")
            with open(save_path, "rb") as f:
                wb_act_dict[ds_name] = pickle.load(f)

        else:
            if model is None:
                raise RuntimeError(
                    f"WB activations for '{ds_name}' not cached and model is not loaded. "
                    "Run Step 2 first."
                )
            pbar.set_postfix(dataset=ds_name, source="extracting...")
            df = dataset_dict[ds_name]
            df_train = df[df["split"] == "train"].reset_index(drop=True)
            df_val   = df[df["split"] == "validation"].reset_index(drop=True)
            bs = 28 if ds_name == "imdb" else 64

            raw_train = get_activations_for_dataset(df_train, model, tokenizer, device=DEVICE, batch_size=bs, show_progress=True)
            raw_val   = get_activations_for_dataset(df_val,   model, tokenizer, device=DEVICE, batch_size=bs, show_progress=True)

            def to_layer_arrays(raw):
                X = {layer: np.array([item["activations"][layer] for item in raw]) for layer in LAYER_LIST}
                y = np.array([item["label"] for item in raw])
                return {"X": X, "y": y}

            payload = {"train": to_layer_arrays(raw_train), "validation": to_layer_arrays(raw_val)}
            with open(save_path, "wb") as f:
                pickle.dump(payload, f)
            wb_act_dict[ds_name] = payload
            gc.collect()

        pbar.update(1)

print(f"\n✅ Step 3 Complete: WB activations ready for {len(wb_act_dict)} datasets in {time.time()-start:.1f}s")


# 4. Load BB Level 2 Features

In [ ]:
l2_feat_dict = {}

print("Step 4: Loading BB Level 2 features...")
start = time.time()

with tqdm(total=len(DATASETS), desc="Step 4 — L2 Features", unit="dataset") as pbar:
    for ds_name in DATASETS:
        local_path = os.path.join(L2_FEAT_DIR, f"{ds_name}_bb_features_lvl2.pkl")
        e2_path    = os.path.join(E2_CACHE,    f"{ds_name}_bb_features_efficiency.pkl")

        if os.path.exists(local_path):
            pbar.set_postfix(dataset=ds_name, source="local cache")
            with open(local_path, "rb") as f:
                data = pickle.load(f)
            X_train, y_train = data["train"]
            X_val,   y_val   = data["validation"]

        elif os.path.exists(e2_path):
            pbar.set_postfix(dataset=ds_name, source="E2 cache")
            with open(e2_path, "rb") as f:
                data = pickle.load(f)
            X_train, y_train = data["train_full"]
            X_val,   y_val   = data["validation"]

        else:
            print(f"  ⚠ No cached L2 features found for {ds_name} — skipping.")
            pbar.update(1)
            continue

        l2_feat_dict[ds_name] = {
            "X_train": X_train, "y_train": y_train,
            "X_val":   X_val,   "y_val":   y_val,
        }
        pbar.update(1)

print(f"\n✅ Step 4 Complete: L2 features loaded for {len(l2_feat_dict)}/17 datasets in {time.time()-start:.1f}s")


# 5. Cross-Dataset Evaluation

For each of the 9 methods (1 BB L2 + 8 WB), evaluate each source classifier on all 17 target datasets and compute recovered metrics.

In [ ]:
# Load E3 in-distribution baselines
df_l2_indist = pd.read_csv(os.path.join(E3_DIR, "level2_results.csv"))
df_l3_indist = pd.read_csv(os.path.join(E3_DIR, "level3_results.csv"))

# Build lookup: (source_dataset, method_key) → best_layer
best_layer_lookup = {}
for _, row in df_l3_indist.iterrows():
    key = (row["Dataset"], METHOD_KEYS.get(row["Method"], row["Method"]))
    best_layer_lookup[key] = int(row["Layer"])

print(f"Loaded E3 baselines: {len(df_l2_indist)} L2 rows, {len(df_l3_indist)} L3 rows")
print(f"Best-layer lookup entries: {len(best_layer_lookup)}")


In [ ]:
results = []
os.makedirs("outputs", exist_ok=True)

available_datasets = [ds for ds in DATASETS if ds in l2_feat_dict and ds in wb_act_dict]
n_ds       = len(available_datasets)
n_methods  = len(WB_METHODS) + 1          # 8 WB + 1 BB L2
n_computed = n_ds * (n_ds - 1) * n_methods
print(f"153 classifiers x 16 targets = {n_ds * (n_ds-1) * n_methods} off-diagonal evaluations")
print(f"Diagonal ({n_ds * n_methods} entries) filled from E3 baselines — no recomputation.")
start = time.time()

with tqdm(total=n_ds, desc="Step 5 — Cross-dataset eval", unit="source") as outer_pbar:
    for source_ds in available_datasets:
        outer_pbar.set_postfix(source=source_ds)

        # Source train activations — used once per source for unsupervised sign correction
        X_src_train_by_layer = wb_act_dict[source_ds]["train"]["X"]
        y_src_train          = wb_act_dict[source_ds]["train"]["y"]

        # ── BB L2: load bundle ONCE per source ────────────────────────────
        l2_clf_path  = os.path.join(L2_CLF_DIR, f"{source_ds}_l2_classifier.pkl")
        l2_in_row    = df_l2_indist[df_l2_indist["Dataset"] == source_ds]
        l2_bundle, in_l2 = None, None

        if os.path.exists(l2_clf_path) and not l2_in_row.empty:
            with open(l2_clf_path, "rb") as f:
                l2_bundle = pickle.load(f)
            in_l2 = {k: l2_in_row[k].values[0] for k in ["AUC", "MAP", "BRP_90"]}
            # Diagonal entry — in-dist = out-dist by definition
            results.append({
                "Method": "BB_L2", "Source": source_ds, "Target": source_ds,
                "InDist_AUC": in_l2["AUC"],    "InDist_MAP": in_l2["MAP"],    "InDist_BRP_90": in_l2["BRP_90"],
                "OutDist_AUC": in_l2["AUC"],   "OutDist_MAP": in_l2["MAP"],   "OutDist_BRP_90": in_l2["BRP_90"],
                "Recovered_AUC": 1.0, "Recovered_MAP": 1.0, "Recovered_BRP_90": 1.0,
            })

        # ── WB: load each probe ONCE per (source, method) ─────────────────
        # key -> (probe_object, best_layer, in_dist_metrics)
        loaded_probes = {}

        for method_key in WB_METHODS:
            probe_path = os.path.join(L3_CLF_DIR, f"{source_ds}_{method_key}_l3_classifier.pkl")
            best_layer = best_layer_lookup.get((source_ds, method_key))
            l3_in_rows = df_l3_indist[
                (df_l3_indist["Dataset"] == source_ds) &
                (df_l3_indist["Method"]  == METHOD_NAMES[method_key])
            ]
            if not os.path.exists(probe_path) or best_layer is None or l3_in_rows.empty:
                continue

            with open(probe_path, "rb") as f:
                probe = pickle.load(f)
            in_wb = {k: l3_in_rows[k].values[0] for k in ["AUC", "MAP", "BRP_90"]}
            loaded_probes[method_key] = (probe, best_layer, in_wb)

            # Diagonal entry for this (source, method)
            results.append({
                "Method": METHOD_NAMES[method_key], "Source": source_ds, "Target": source_ds,
                "Layer": best_layer,
                "InDist_AUC": in_wb["AUC"],    "InDist_MAP": in_wb["MAP"],    "InDist_BRP_90": in_wb["BRP_90"],
                "OutDist_AUC": in_wb["AUC"],   "OutDist_MAP": in_wb["MAP"],   "OutDist_BRP_90": in_wb["BRP_90"],
                "Recovered_AUC": 1.0, "Recovered_MAP": 1.0, "Recovered_BRP_90": 1.0,
            })

        # ── Off-diagonal: one pass over targets, all classifiers already in memory
        for target_ds in available_datasets:
            if target_ds == source_ds:
                continue

            y_t = wb_act_dict[target_ds]["validation"]["y"]

            # BB L2
            if l2_bundle is not None:
                out_l2 = evaluate_bb_on_target(l2_bundle, l2_feat_dict[target_ds]["X_val"], y_t)
                rec_l2 = compute_recovered_metrics(in_l2, out_l2)
                results.append({
                    "Method": "BB_L2", "Source": source_ds, "Target": target_ds,
                    "InDist_AUC": in_l2["AUC"],       "InDist_MAP": in_l2["MAP"],       "InDist_BRP_90": in_l2["BRP_90"],
                    "OutDist_AUC": out_l2["AUC"],      "OutDist_MAP": out_l2["MAP"],     "OutDist_BRP_90": out_l2["BRP_90"],
                    **rec_l2,
                })

            # WB probes — each already loaded once above
            for method_key, (probe, best_layer, in_wb) in loaded_probes.items():
                X_t_wb = wb_act_dict[target_ds]["validation"]["X"][best_layer]
                out_wb = evaluate_wb_on_target(
                    probe, method_key,
                    X_src_train_by_layer[best_layer], y_src_train,
                    X_t_wb, y_t,
                )
                rec_wb = compute_recovered_metrics(in_wb, out_wb)
                results.append({
                    "Method": METHOD_NAMES[method_key], "Source": source_ds, "Target": target_ds,
                    "Layer": best_layer,
                    "InDist_AUC": in_wb["AUC"],      "InDist_MAP": in_wb["MAP"],      "InDist_BRP_90": in_wb["BRP_90"],
                    "OutDist_AUC": out_wb["AUC"],    "OutDist_MAP": out_wb["MAP"],    "OutDist_BRP_90": out_wb["BRP_90"],
                    **rec_wb,
                })

        outer_pbar.update(1)

df_results = pd.DataFrame(results)
df_results.to_csv("outputs/generalisation_results.csv", index=False)
elapsed = time.time() - start
print(f"\n✅ Step 5 Complete: {len(df_results)} rows in {elapsed:.1f}s")
print(f"   ({n_computed} computed  +  {n_ds * n_methods} diagonal)  →  outputs/generalisation_results.csv")


# 6. Recovery Heatmaps

Three heatmaps — one per metric — each showing the **average recovered metric across all 9 methods**. Diagonal is always 100% (in-distribution). Colour: green = high recovery, red = low recovery.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_results = pd.read_csv("outputs/generalisation_results.csv")

datasets_ordered = [ds for ds in DATASETS if ds in df_results["Source"].unique()]

for metric_key, metric_label, title in [
    ("Recovered_AUC",    "Recovered AUC",    "Cross-Dataset Generalisation — Recovered AUC (avg. all methods)"),
    ("Recovered_MAP",    "Recovered MAP",    "Cross-Dataset Generalisation — Recovered MAP (avg. all methods)"),
    ("Recovered_BRP_90", "Recovered BRP_90", "Cross-Dataset Generalisation — Recovered BRP_90 (avg. all methods)"),
]:
    avg = (
        df_results
        .groupby(["Source", "Target"])[metric_key]
        .mean()
        .unstack("Target")
        .reindex(index=datasets_ordered, columns=datasets_ordered)
    )

    save_path = f"outputs/{metric_key.lower()}_heatmap.png"
    plot_recovery_heatmap(avg, metric_label, title, save_path)

print("\n✅ Step 6 Complete: 3 heatmaps saved to outputs/")
